# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This ensures reproducibility and allows standardized access to structured metadata and tabular data.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their respective `@id`s.

The Croissant schema organizes data into multiple "record sets." Each record set corresponds to a structured table in the dataset and contains fields (columns) uniquely referenced by their `@id`.

Let's inspect the available record sets and their structure.

In [ ]:
# Collect available record sets using their @id
record_sets = []
fields_map = {}

for rs in metadata.recordSet:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    record_sets.append(rs_id)
    # Each record set contains fields, which are also referenced by @id
    fields = rs.get('field', []) if isinstance(rs, dict) else []
    if isinstance(fields, dict):
        # Sometimes single dict
        fields = [fields]
    fields_map[rs_id] = []
    for f in fields:
        field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
        fields_map[rs_id].append(field_id)

print("Record sets available:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs}")

for rs in record_sets:
    print(f"\nFields in RecordSet {rs}:")
    for fid in fields_map.get(rs, []):
        print(f"  * Field @id: {fid}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We reference record sets and fields exclusively by their `@id` to ensure robust and reproducible access.

In [ ]:
# Extract data from each available record set
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"\nColumns in record set {record_set}:")
        print(df.columns.tolist())
        print(df.head())

# Select a specific record set for demonstration
if record_sets:
    selected_record_set = record_sets[0]
    print(f"\nPreview of first record set (@id={selected_record_set}):")
    print(dataframes[selected_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply data processing steps to one record set.

We'll filter records based on numeric criteria, normalize a numeric field, and group data by a categorical attribute using their exact `@id` references.

Replace `<numeric_field_id>` and `<group_field_id>` below with the actual `@id` values from the previous section, as needed.

In [ ]:
# Choose a record set and numeric/group fields to operate on
# For illustration, we use the first available record set and its first numeric field
rs_id = selected_record_set
df = dataframes[rs_id]

# Try to identify a numeric field for analysis (e.g., 'age')
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    elif 'clitomegaly' in col.lower() or 'hirsutism' in col.lower():
        group_field_id = col

print(f"Using numeric field: {numeric_field_id} and group field: {group_field_id}")

# Filtering by an arbitrary threshold (customize as per dataset)
if numeric_field_id is not None:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a field
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
We visualize data distributions and relationships between fields in the selected record set.

Let's plot the distribution of the numeric field and its normalized values.

In [ ]:
if numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    filtered_df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet: {rs_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(8, 4))
    filtered_df[f'{numeric_field_id}_normalized'].hist(bins=10)
    plt.title(f"Normalized Distribution of {numeric_field_id} (RecordSet: {rs_id})")
    plt.xlabel(f'{numeric_field_id}_normalized')
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` for reproducibility.

- The dataset comprises structured clinical and genetic tables for three female patients with NC-21OHD-associated infertility.
- Metadata and tabular data are accessible via Croissant, making FAIR-aligned exploration and analysis straightforward.
- Numeric and categorical fields were filtered, normalized, grouped, and visualized.

Further analyses can extend to other record sets and fields, supporting academic research and case-based exploration.